In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import brainunit as u
from braincell import Soma, Axon, BasalDendrite, ApicalDendrite, Branch, Morpho


# `morpho`: from `Branch` geometry to editable morphology trees

This notebook walks through the current morphology user flow in `braincell`:

1. Build single-branch geometry with `Branch`
2. Assemble an editable tree with `Morpho` and `MorphoBranch`
3. Query topology and tree-level metrics
4. Import real morphologies from `SWC` and `ASC` files


## 1. `Branch`: the smallest geometry primitive

`Branch` describes one anatomical branch as an ordered sequence of frustum segments. To build it, you need segment lengths or 3D points, plus proximal and distal radii.

This notebook starts with `Branch.from_lengths(...)`. It supports two radius styles:

- Paired radii: provide `radii_proximal` and `radii_distal` with one value per segment
- Shared radii: provide `radii` with `N + 1` node radii for `N` connected segments


In [ ]:
# Equivalent single-segment constructions
soma_0 = Branch.from_lengths(
    lengths=[20.0] * u.um,
    radii_proximal=[10.0] * u.um,
    radii_distal=[10.0] * u.um,
    type = 'soma'
)

soma_1 = Soma.from_lengths(
    lengths=[20.0] * u.um,
    radii_proximal=[10.0] * u.um,
    radii_distal=[10.0] * u.um,
)
soma_2 = Soma.from_lengths(
    lengths=[20.0] * u.um,
    radii=[10.0, 10.0] * u.um,
)

print("soma_0 == soma_1:", soma_0 == soma_1)
print("soma_0 == soma_2:", soma_0 == soma_2)
print(soma_0)
soma_0.vis2d()

# The same idea extends to multi-segment branches
basal_0 = BasalDendrite.from_lengths(
    lengths=[10.0, 20.0] * u.um,
    radii_proximal=[3.0, 2.0] * u.um,
    radii_distal=[2.0, 1.0] * u.um,
)
basal_1 = BasalDendrite.from_lengths(
    lengths=[10.0, 20.0] * u.um,
    radii=[3.0, 2.0, 1.0] * u.um,
)
print("basal_0 == basal_1:", basal_0 == basal_1)
print(basal_0)
basal_0.vis2d()

In [ ]:
## branch.from_points
soma_3 = Soma.from_points(
    points=[(0.0, 0.0, 0.0), (20.0, 0.0, 0.0)] * u.um,
    radii_proximal=[10.0] * u.um,
    radii_distal=[10.0] * u.um,
)
soma_3.vis3d(jupyter_backend='html')

### Why keep both radius styles?

When radii stay continuous across segment boundaries, shared `radii=[r0, ..., rN]` is compact and easy to read.

When a branch contains a radius jump, the paired form is more explicit:

- use `radii_proximal` and `radii_distal` to describe the discontinuity directly
- the shared-radius form can still represent the same geometry, but it needs an extra zero-length segment at the jump

This behavior is intentional. Real morphology files sometimes contain coincident points with different radii to encode an abrupt diameter change, and `Branch` preserves that geometry.


In [ ]:
jump_0 = Branch.from_lengths(
    lengths=[10.0, 20.0] * u.um,
    radii_proximal=[3.0, 1.0] * u.um,
    radii_distal=[2.0, 1.0] * u.um,
)
jump_1 = Branch.from_lengths(
    lengths=[10.0, 0.0, 20.0] * u.um,
    radii=[3.0, 2.0, 1.0, 1.0] * u.um,
)
print("jump_0 == jump_1:", jump_0 == jump_1)
jump_0.vis2d()

The table below summarizes the most useful `Branch` attributes. If a branch is created with `from_lengths(...)`, point-related fields are `None`. Use `from_points(...)` when later steps need spatial coordinates, such as spatial ranges, projections, or Euclidean distances.

| Attribute | Description |
| --- | --- |
| lengths | Length of each frustum segment (`N` values) |
| length | Total branch length |
| areas | Surface area of each segment (`N` values) |
| area | Total surface area of the branch |
| volumes | Volume of each segment (`N` values) |
| volume | Total branch volume |
| mean_radius | Length-weighted mean radius |
| radii_proximal | Proximal radius of each segment (`N` values) |
| radii_distal | Distal radius of each segment (`N` values) |
| radii | Radii at all shared nodes (`N + 1` values) when boundaries are continuous |
| points_proximal | Start point of each segment (`N` points) |
| points_distal | End point of each segment (`N` points) |
| points | Shared node coordinates (`N + 1` points) when point geometry is continuous |
| type | Branch type (`axon`, `dendrite`, `soma`, and related variants) |
| n_segments | Number of frustum segments |


In [ ]:
# Reusable branches for the rest of the notebook
soma = Soma.from_lengths(
    lengths=[20.0] * u.um,
    radii=[10.0, 10.0] * u.um,
)
basal = BasalDendrite.from_lengths(
    lengths=[40.0, 30.0] * u.um,
    radii=[2.0, 1.8, 1.2] * u.um,
)
apical = ApicalDendrite.from_points(
    points=[(0.0, 0.0, 0.0), (15.0, 0.0, 0.0), (30.0, 5.0, 0.0)] * u.um,
    radii=[1.5, 1.2, 0.9] * u.um,
)
axon = Axon.from_points(
    points=[(-30.0, -5.0, 0.0), (-12.0, -2.0, 0.0), (-4.0, 0.0, 0.0)] * u.um,
    radii=[0.8, 0.9, 1.0] * u.um,
)

for branch in [soma, basal, apical, axon]:
    print(branch)
    branch.vis2d()


## 2. `Morpho` / `MorphoBranch`: tree construction, attachment, and naming

`Morpho` owns the whole editable tree. `MorphoBranch` is a tree-local view of one branch, so it exposes branch geometry together with parent/child relationships.

This section demonstrates three common patterns:

- `Morpho.from_root(...)` to create the root branch
- `tree.soma.xxx = branch` syntax sugar for child attachment
- `tree.attach(...)` or `tree.soma[0.5, 1.0].xxx = branch` for explicit `parent_x` / `child_x`

Notes:

- `parent_x` must be one of `0`, `0.5`, or `1`
- `parent_x = 0.5` is only allowed when the parent branch type is `soma`
- `child_x` currently accepts only `0` or `1`
- if you do not provide an explicit child name, `Morpho` auto-generates one as `type_N`


In [ ]:
# Start from the root branch
tree = Morpho.from_root(soma, name="soma")

# Syntax sugar: the attribute name becomes the child branch name
tree.soma.basal_slot = basal
print(tree.metric)
tree.vis2d(mode='frustum')

In [ ]:
# Explicit attach: without child_name, Morpho allocates type_N automatically
tree.attach(parent="soma", child_branch=apical, parent_x=0.5)
tree.vis2d(mode='frustum')

In [ ]:
# Attachment-point syntax makes parent_x and child_x visible at the call site
tree.soma[1.0, 1.0].axon_back = axon
tree.vis2d(
    mode='frustum',   
    layout_family="balloon",)
print(tree.topo())

## 3. Queries, summaries, and tree-level metrics

Once the tree is built, `Morpho` supports structural queries, a compact summary helper, and derived metrics.

Common queries:

- `tree.branches`
- `tree.branches_by_order(order=...)`
- `tree.branch(name=...)`
- `tree.branch(index=..., order=...)`
- `tree.path_to_root(...)`
- `tree.summary()`

Common metrics:

- `n_branches` / `n_stems` / `n_bifurcations` / `max_branch_order`
- `total_length` / `total_area` / `total_volume` / `mean_radius`
- when `tree.has_full_point_geometry` is `True`, `x_range / y_range / z_range`, `max_euclidean_distance`, and `vis3d(...)` are available

`tree.summary()` is meant to stay compact even for larger morphologies, so it focuses on counts, aggregate geometry, and whether the full tree carries 3D point geometry.

For coordinate-dependent metrics such as `x_range` and `max_euclidean_distance`, and for `tree.vis3d(...)`, the tree should satisfy `tree.has_full_point_geometry == True`. Mixed trees are still fine for topology and cable-style aggregate metrics, but spatial measurements now deliberately require full point coverage.


In [ ]:
point_soma = Branch.from_points(
    points=[(0.0, 0.0, 0.0), (10.0, 0.0, 0.0)] * u.um,
    radii=[5.0, 5.0] * u.um,
    type="soma",
)
point_main = Branch.from_points(
    points=[(5.0, 0.0, 0.0), (5.0, 10.0, 0.0)] * u.um,
    radii=[2.0, 1.5] * u.um,
    type="basal_dendrite",
)
point_tuft = Branch.from_points(
    points=[(5.0, 10.0, 0.0), (5.0, 20.0, 0.0)] * u.um,
    radii=[1.5, 1.0] * u.um,
    type="apical_dendrite",
)
point_side = Branch.from_points(
    points=[(10.0, 0.0, 0.0), (12.0, 0.0, 0.0)] * u.um,
    radii=[1.0, 0.8] * u.um,
    type="axon",
)

point_tree = Morpho.from_root(point_soma, name="soma")
point_tree.soma.attach(point_main, name="main", parent_x=0.5)
point_tree.main.attach(point_tuft, name="tuft")
point_tree.soma.attach(point_side, name="side", parent_x=1.0)

## 4. `SWC` import: recommended entry point and report

`Morpho.from_swc(...)` is the main file-import entry point.

You can load just the tree:

- `tree = Morpho.from_swc(path)`

Or load the tree together with a structured reader report:

- `tree, report = Morpho.from_swc(path, return_report=True)`

The real SWC fixtures in this repository often produce warnings such as reordered nodes, renumbered indices, or contour detection. Those warnings usually describe normalization steps rather than fatal import failures.


In [ ]:
repo_root = Path.cwd()
if not (repo_root / "braincell/io/morpho_files").exists():
    repo_root = repo_root.parent
fixture_dir = repo_root / "braincell/io/morpho_files"

swc_tree, swc_report = Morpho.from_swc(fixture_dir / "io.swc", return_report=True)

If you want a fixture with more visible warnings, load `bc.swc`.

This file triggers warnings for duplicate `xyzr` samples and unknown node types, but the reader can still normalize it into a usable `Morpho`.


In [ ]:
warn_tree, warn_report = Morpho.from_swc(fixture_dir / "bc.swc", return_report=True)


## 5. `ASC`: minimal working example

The `ASC` entry point mirrors `SWC`:

- `tree = Morpho.from_asc(path)`
- `tree, report = Morpho.from_asc(path, return_report=True)`

This section only checks that the current reader builds a `Morpho` successfully; it does not dive into the parsing rules.


In [ ]:
asc_tree, asc_report = Morpho.from_asc(fixture_dir / "goc.asc", return_report=True)


## 7. Summary

At this point you have seen the main entry points in the current `morpho` layer:

- build geometry with `Branch.from_lengths(...)` and `Branch.from_points(...)`
- organize a full tree with `Morpho.from_root(...)`, syntax sugar attachment, and `attach(...)`
- query structure with `branch(...)`, `branches_by_order(...)`, `path_to_root(...)`, and tree metrics
- import real morphologies with `Morpho.from_swc(...)` and `Morpho.from_asc(...)`
- compare manual construction against file import with `example_tree.swc`

If you want to continue:

- for more file-format semantics, read `develop_doc/io.md`
- for how morphology becomes a discrete cell view, read `develop_doc/cell.ipynb`
